# CURE-Rec — behavioral calibration robustness

This notebook runs the **pre-specified CURE-Sim assumption-sensitivity study** after the final BPR archive is frozen. It does not use MovieLens to calibrate causal effects and it does not tune the recommendation model.

Each enabled run recomputes the exact six-player game for every configuration and seed. Leave all execution guards `False` until you deliberately want that cost.


## 1. Setup

Run after `git pull` and a kernel restart. The cell resolves the CURE-Rec source directory from either the repository root or the code directory.


In [ ]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd

CWD = Path.cwd().resolve()
CANDIDATES = [CWD, CWD / 'paper-ideas' / 'CURE-Rec' / 'code', *CWD.parents]
ROOT = next((p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the CURE-Rec code directory or repository root.')
sys.path[:] = [str(ROOT), *[item for item in sys.path if item != str(ROOT)]]
for name in list(sys.modules):
    if name == 'cure_rec' or name.startswith('cure_rec.'):
        del sys.modules[name]
importlib.invalidate_caches()

from cure_rec.calibration import run_calibration_sweep
from cure_rec.config import load_settings

QUICK_CONFIG = ROOT / 'configs' / 'curesim_quickstart.yaml'
FULL_CONFIG = ROOT / 'configs' / 'curesim_full.yaml'
RUN_ROOT = ROOT / 'runs'
print('CURE-Rec source:', ROOT)


## 2. Read this before enabling a run

- **OAT** runs one frozen baseline and two predeclared contrast values for each of seven assumptions: 15 configurations total.
- **LHS** runs one frozen baseline plus the requested number of joint Latin-hypercube configurations.
- Each configuration is repeated over the stated independent environment seeds and evaluates all 64 coalitions in every configured scenario.
- Neither design selects a best configuration. All seed-level results, feasibility outcomes, planner modes, attributions, and interactions are retained.


In [ ]:
# All costly actions are disabled by default. Enable exactly one action at a time.
RUN_CALIBRATION_SMOKE = False
RUN_CALIBRATION_OAT_FULL = False
RUN_CALIBRATION_LHS_FULL = False

# Full inference uses the same five independent seeds as the accepted full sweep.
FULL_SEEDS = (42, 43, 44, 45, 46)
LHS_SAMPLES = 24

assert sum((RUN_CALIBRATION_SMOKE, RUN_CALIBRATION_OAT_FULL, RUN_CALIBRATION_LHS_FULL)) <= 1


## 3. Action 1 — cheap structural smoke run

This is still an exact-game run, but uses the quick simulator, one seed, and one LHS configuration plus the baseline. It verifies output tables/figures before the full study.


In [ ]:
if RUN_CALIBRATION_SMOKE:
    smoke = load_settings(QUICK_CONFIG)
    smoke.run.name = 'calibration-smoke'
    smoke.run.output_root = RUN_ROOT
    smoke_result = run_calibration_sweep(smoke, seeds=(42,), design='lhs', lhs_samples=1)
    print('Smoke calibration:', smoke_result.run_dir)
    display(smoke_result.summary)
else:
    print('Smoke calibration disabled.')


## 4. Action 2 — full one-at-a-time phase diagram

This is the interpretable primary sensitivity analysis. It evaluates 15 configurations × 5 seeds under the full four-scenario CURE-Sim configuration. Expect a long run; do not interrupt it for notebook formatting changes.


In [ ]:
if RUN_CALIBRATION_OAT_FULL:
    full = load_settings(FULL_CONFIG)
    full.run.name = 'calibration-oat-full'
    full.run.output_root = RUN_ROOT
    oat_result = run_calibration_sweep(full, seeds=FULL_SEEDS, design='oat')
    print('Full OAT calibration:', oat_result.run_dir)
    display(oat_result.summary)
else:
    print('Full OAT calibration disabled.')


## 5. Action 3 — joint Latin-hypercube robustness probe

Run only after the OAT phase diagram has completed. This examines joint assumptions rather than picking a favorable setting. With `LHS_SAMPLES = 24`, it evaluates 25 configurations (including the baseline) × 5 seeds.


In [ ]:
if RUN_CALIBRATION_LHS_FULL:
    full = load_settings(FULL_CONFIG)
    full.run.name = 'calibration-lhs-full'
    full.run.output_root = RUN_ROOT
    lhs_result = run_calibration_sweep(full, seeds=FULL_SEEDS, design='lhs', lhs_samples=LHS_SAMPLES)
    print('Full LHS calibration:', lhs_result.run_dir)
    display(lhs_result.summary)
else:
    print('Full LHS calibration disabled.')


## 6. Action 4 — inspect a completed calibration run

Paste the run directory printed by Action 2 or 3. This operation is cheap and never re-runs coalitions.


In [ ]:
CALIBRATION_OUTPUT = None  # Example: RUN_ROOT / 'calibration-oat-YYYYMMDDTHHMMSSZ'

if CALIBRATION_OUTPUT is not None:
    CALIBRATION_OUTPUT = Path(CALIBRATION_OUTPUT)
    calibration_summary = pd.read_csv(CALIBRATION_OUTPUT / 'calibration_summary.csv')
    calibration_configurations = pd.read_csv(CALIBRATION_OUTPUT / 'calibration_configurations.csv')
    calibration_manifest = json.loads((CALIBRATION_OUTPUT / 'calibration_manifest.json').read_text())
    display(calibration_summary)
    display(calibration_configurations)
    print(json.dumps({
        'design': calibration_manifest['design'],
        'seeds': calibration_manifest['seeds'],
        'base_config_hash': calibration_manifest['base_config_hash'],
        'figures': sorted(path.name for path in (CALIBRATION_OUTPUT / 'figures').glob('*.png')),
    }, indent=2))
else:
    print('Set CALIBRATION_OUTPUT after a calibration run to inspect its artifacts.')


## Interpretation gate

Report this study as sensitivity of the disclosed CURE-Sim behavioral model. A stable `repeat_cap` selection rate across these settings supports simulator robustness; it does **not** turn MovieLens ratings into logged causal policy evidence. Retain configurations with unfavorable results, feasibility failures, and repair decisions.
